In [ ]:
import os
import warnings
import gc
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'

import tensorflow as tf

tf.get_logger().setLevel('ERROR')
tf.get_logger().setLevel("ERROR")
warnings.simplefilter("ignore")

from optuna.storages import RetryFailedTrialCallback
import json
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import GlobalAveragePooling2D, multiply, Conv2D, Dense, BatchNormalization, LayerNormalization, Input, MaxPool2D, Dropout, Flatten
import numpy as np
import keras
from phd.Augmentations.common_code.augmentation_functions import *
from phd.Augmentations.common_code.get_dataset import *
from phd.Augmentations.common_code.models import *
from phd.Augmentations.common_code.training_loop import *
from phd.Augmentations.common_code.utils import *
import optuna
from astropy.modeling.models import Sersic2D
from astropy.visualization import PercentileInterval

AUTO = tf.data.AUTOTUNE


import wandb

wandb.login()



config_name = input("Enter config:")

configs = {
    "SDSS": {
        "train_size": 272583,
        "test_size": 30287,
        "bin_folder_name": "Bin_SDSS_z_0.0-0.25",
        "experiment_name": "Bin_SDSS"
    },
    "DES": {
        "train_size": 101193,
        "test_size": 11244,
        "bin_folder_name": "Bin_DES_z_0.0-0.15",
        "experiment_name": "Bin_DES"
    },
    "H1": {
        "train_size": 5863,
        "test_size": 655,
        "bin_folder_name": "Bin_Hubble_z_0.0-0.3",
        "experiment_name": "Bin_H_1"
    },
    "H2": {
        "train_size": 17811,
        "test_size": 1983,
        "bin_folder_name": "Bin_Hubble_z_0.3-1.0",
        "experiment_name": "Bin_H_2"
    },
    "H3": {
        "train_size": 17038,
        "test_size": 1893,
        "bin_folder_name": "Bin_Hubble_z_1.0_4.0",
        "experiment_name": "Bin_H_3"
    }
}

config = configs[config_name]
train_size = config["train_size"]
test_size = config["test_size"]
bin_folder_name = config["bin_folder_name"]
experiment_name = config["experiment_name"]

training_config = {
"ENABLE_DETERMINISM": False,
"SEED": 17,
"AUTO": tf.data.AUTOTUNE,
"TRAIN_BATCH_SIZE": 128,
"TEST_BATCH_SIZE": 128,
"NUMBER_OF_EPOCHS": 500,
"EARLY_STOPPING_TOLERANCE": 10,
"LEARNING_RATE": 1e-4,
"IMAGE_SIZE": 128,
"SHUFFLE_BUFFER": train_size,
"TPU": False,
"AUGMENTATIONS_ROTATE": False,
"AUGMENTATIONS_FLIP_HORIZONTALLY": False,
"AUGMENTATIONS_FLIP_VERTICALLY": False,
"AUGMENTATIONS_ZOOM": False,
"AUGMENTATIONS_RANDOM_NOISE": False,
"AUGMENTATIONS_CENTER_NOISE": False,
"AUGMENTATIONS_OUTSIDE_NOISE": False,
"AUGMENTATIONS_PERLIN_CENTER_NOISE": False,
"AUGMENTATIONS_SALT_AND_PEPPER": False,
"AUGMENTATIONS_CUTOUT": False,
"USE_ADABELIEF_OPTIMIZER": False,
"GALAXY_DATASET": "SDSS",
"TRAIN_DATASET_SIZE": train_size,
"TEST_DATASET_SIZE": test_size,
"STRETCH": False,
"STEPS_PER_EXECUTION": 1,
"FOLDS": 10,
"NUMBER_OF_CLASSES": 2,
"CACHE": False,
"METRIC": "F1",
"LOAD_WEIGHTS": False
}
training_config["WANDB_PROJECT_NAME"] = f"Paper3, Step 3"
training_config["EXPERIMENT_DESCRIPTION"] = f"paper3_focal_last_{experiment_name}"
training_config["LOCAL_GCP_PATH_BASE"] = f"."
training_config["REMOTE_GCP_PATH_BASE"] = f"."
training_config["DATASET_PATH"] = f"TFDataset/{bin_folder_name}/all"


@keras.saving.register_keras_serializable()
class FocalConv(keras.layers.Layer):
    def __init__(self, filters, kernel_size, factor, activation):
        super().__init__()
        self.filters = filters
        self.kernel = kernel_size
        self.activation = activation
        self.factor = factor
        self.conv1 = Conv2D(filters=self.filters, kernel_size=self.kernel, activation=self.activation, padding="same")

    def build(self, input_shape):
        self.gauss = self.make_gaussian(size=input_shape[1], depth=self.filters, factor=self.factor)

    def call(self, x, training=False):
      x = self.conv1(x, training=training)
      batch_size = tf.shape(x)[0]
      xx = tf.repeat(self.gauss, repeats=batch_size, axis=0)
      x = multiply([x, xx])

      return x
    
    def get_config(self):
       return {
          "filters": self.filters,
          "kernel_size": self.kernel,
          "activation": self.activation,
          "factor": self.factor
       }

    def make_gaussian(self, size: int, depth: int, factor: float):
        """ Make a square gaussian kernel.

        size is the length of a side of the square
        fwhm is full-width-half-maximum, which
        can be thought of as an effective radius.
        """
        fwhm = size // 2
        x = np.arange(0, size, 1, float)
        y = x[:,np.newaxis]

        x0 = y0 = size // 2

        gauss = np.exp(-1 * factor * np.log(2) * ((x-x0) ** 2 + (y-y0) ** 2) / fwhm **2)

        return tf.expand_dims(np.stack([gauss for _ in range(depth)], -1), axis=0)





def cavanagh_no_focal(training_config):
  model = Sequential(name="Cavanagh")
  model.add(Input(shape=(training_config["IMAGE_SIZE"], training_config["IMAGE_SIZE"], 3)))
    
  model.add(Conv2D(filters=32, kernel_size=7, activation='relu', padding='same'))
  model.add(BatchNormalization())
  model.add(MaxPool2D(2))

  model.add(Conv2D(filters=64, kernel_size=5, activation='relu', padding='same'))
  model.add(Conv2D(filters=64, kernel_size=5, activation='relu', padding='same'))
  model.add(BatchNormalization())
  model.add(MaxPool2D(2))

  model.add(Conv2D(filters=128, kernel_size=3, activation='relu', padding='same'))
  model.add(BatchNormalization())
  model.add(MaxPool2D(2))

  model.add(Flatten())
  model.add(Dropout(0.5))

  model.add(Dense(256, activation='relu'))
  model.add(Dense(256, activation='relu'))
  model.add(Dense(training_config["NUMBER_OF_CLASSES"], activation='softmax'))

  return model

def cavanagh_focal_10(training_config):
  model = Sequential(name="Cavanagh")
  model.add(Input(shape=(training_config["IMAGE_SIZE"], training_config["IMAGE_SIZE"], 3)))

  model.add(FocalConv(filters=32, kernel_size= 7, factor = 10, activation='relu'))
  model.add(BatchNormalization())
  model.add(MaxPool2D(2))

  model.add(FocalConv(filters=64, kernel_size= 5, factor = 10, activation='relu'))
  model.add(FocalConv(filters=64, kernel_size= 5, factor = 10, activation='relu'))
  model.add(BatchNormalization())
  model.add(MaxPool2D(2))

  model.add(FocalConv(filters=128, kernel_size=3, factor = 10, activation='relu'))
  model.add(BatchNormalization())
  model.add(MaxPool2D(2))

  model.add(Flatten())
  model.add(Dropout(0.5))

  model.add(Dense(256, activation='relu'))
  model.add(Dense(256, activation='relu'))
  model.add(Dense(training_config["NUMBER_OF_CLASSES"], activation='softmax'))

  return model

def top_10_loss_4class_original(training_config):
  model = Sequential(name="Cavanagh")
  model.add(Input(shape=(training_config["IMAGE_SIZE"], training_config["IMAGE_SIZE"], 3)))

  model.add(FocalConv(filters=64, kernel_size= 7, factor = 8, activation='relu'))
  model.add(BatchNormalization())
  model.add(MaxPool2D(2))

  model.add(FocalConv(filters=64, kernel_size= 5, factor = 5, activation='relu'))
  model.add(BatchNormalization())
  model.add(MaxPool2D(2))

  model.add(Conv2D(filters=32, kernel_size=7, activation='relu', padding='same'))
  model.add(BatchNormalization())
  model.add(MaxPool2D(2))

  model.add(Conv2D(filters=32, kernel_size=5, activation='relu', padding='same')) # wave replaced with regular
  model.add(BatchNormalization())
  model.add(MaxPool2D(2))
    
  model.add(Flatten())
  model.add(Dropout(0.5))

  model.add(Dense(32, activation='relu'))
  model.add(Dense(160, activation='relu'))
  model.add(Dense(160, activation='relu'))
  model.add(Dense(training_config["NUMBER_OF_CLASSES"], activation='softmax'))

  return model

def top_10_loss_4class_all_focal(training_config):
  model = Sequential(name="Cavanagh")
  model.add(Input(shape=(training_config["IMAGE_SIZE"], training_config["IMAGE_SIZE"], 3)))

  model.add(FocalConv(filters=64, kernel_size= 7, factor = 10, activation='relu'))
  model.add(BatchNormalization())
  model.add(MaxPool2D(2))

  model.add(FocalConv(filters=64, kernel_size= 5, factor = 10, activation='relu'))
  model.add(BatchNormalization())
  model.add(MaxPool2D(2))

  model.add(FocalConv(filters=32, kernel_size=7, factor = 10, activation='relu'))
  model.add(BatchNormalization())
  model.add(MaxPool2D(2))

  model.add(FocalConv(filters=32, kernel_size=5, factor = 10, activation='relu')) # wave replaced with regular
  model.add(BatchNormalization())
  model.add(MaxPool2D(2))
    
  model.add(Flatten())
  model.add(Dropout(0.5))

  model.add(Dense(32, activation='relu'))
  model.add(Dense(160, activation='relu'))
  model.add(Dense(160, activation='relu'))
  model.add(Dense(training_config["NUMBER_OF_CLASSES"], activation='softmax'))

  return model
    
def top_10_loss_4class_no_focal(training_config):
  model = Sequential(name="Cavanagh")
  model.add(Input(shape=(training_config["IMAGE_SIZE"], training_config["IMAGE_SIZE"], 3)))

  model.add(Conv2D(filters=64, kernel_size= 7, activation='relu', padding='same'))
  model.add(BatchNormalization())
  model.add(MaxPool2D(2))

  model.add(Conv2D(filters=64, kernel_size= 5, activation='relu', padding='same'))
  model.add(BatchNormalization())
  model.add(MaxPool2D(2))

  model.add(Conv2D(filters=32, kernel_size=7, activation='relu', padding='same'))
  model.add(BatchNormalization())
  model.add(MaxPool2D(2))
    
  model.add(Conv2D(filters=32, kernel_size=5, activation='relu', padding='same'))
  model.add(BatchNormalization())
  model.add(MaxPool2D(2))
  
  model.add(Flatten())
  model.add(Dropout(0.5))

  model.add(Dense(32, activation='relu'))
  model.add(Dense(160, activation='relu'))
  model.add(Dense(160, activation='relu'))
  model.add(Dense(training_config["NUMBER_OF_CLASSES"], activation='softmax'))

  return model

def best_loss_11class_original(training_config):
  model = Sequential(name="Cavanagh")
  model.add(Input(shape=(training_config["IMAGE_SIZE"], training_config["IMAGE_SIZE"], 3)))

  model.add(FocalConv(filters=48, kernel_size= 3, factor = 10, activation='relu'))
  model.add(BatchNormalization())
  model.add(MaxPool2D(2))

  model.add(Conv2D(filters=32, kernel_size= 7, activation='relu', padding='same'))
  model.add(BatchNormalization())
  model.add(MaxPool2D(2))

  model.add(Conv2D(filters=48, kernel_size=7, activation='relu', padding='same'))
  model.add(BatchNormalization())
  model.add(MaxPool2D(2))

  model.add(Conv2D(filters=32, kernel_size=3, activation='relu', padding='same')) # wave replaced with regular
  model.add(BatchNormalization())
  model.add(MaxPool2D(2))
    
  model.add(Flatten())
  model.add(Dropout(0.5))

  model.add(Dense(224, activation='relu'))
  model.add(Dense(192, activation='relu'))
  model.add(Dense(training_config["NUMBER_OF_CLASSES"], activation='softmax'))

  return model

def best_loss_11class_all_focal(training_config):
  model = Sequential(name="Cavanagh")
  model.add(Input(shape=(training_config["IMAGE_SIZE"], training_config["IMAGE_SIZE"], 3)))

  model.add(FocalConv(filters=48, kernel_size= 3, factor = 10, activation='relu'))
  model.add(BatchNormalization())
  model.add(MaxPool2D(2))

  model.add(FocalConv(filters=32, kernel_size= 7, factor = 10, activation='relu'))
  model.add(BatchNormalization())
  model.add(MaxPool2D(2))

  model.add(FocalConv(filters=48, kernel_size=7, factor = 10, activation='relu'))
  model.add(BatchNormalization())
  model.add(MaxPool2D(2))

  model.add(FocalConv(filters=32, kernel_size=3, factor = 10, activation='relu'))
  model.add(BatchNormalization())
  model.add(MaxPool2D(2))
    
  model.add(Flatten())
  model.add(Dropout(0.5))

  model.add(Dense(224, activation='relu'))
  model.add(Dense(192, activation='relu'))
  model.add(Dense(training_config["NUMBER_OF_CLASSES"], activation='softmax'))

  return model

def best_loss_11class_no_focal(training_config):
  model = Sequential(name="Cavanagh")
  model.add(Input(shape=(training_config["IMAGE_SIZE"], training_config["IMAGE_SIZE"], 3)))

  model.add(Conv2D(filters=48, kernel_size= 3, activation='relu', padding='same'))
  model.add(BatchNormalization())
  model.add(MaxPool2D(2))

  model.add(Conv2D(filters=32, kernel_size= 7, activation='relu', padding='same'))
  model.add(BatchNormalization())
  model.add(MaxPool2D(2))

  model.add(Conv2D(filters=48, kernel_size=7, activation='relu', padding='same'))
  model.add(BatchNormalization())
  model.add(MaxPool2D(2))

  model.add(Conv2D(filters=32, kernel_size=3, activation='relu', padding='same')) # wave replaced with regular
  model.add(BatchNormalization())
  model.add(MaxPool2D(2))
    
  model.add(Flatten())
  model.add(Dropout(0.5))

  model.add(Dense(224, activation='relu'))
  model.add(Dense(192, activation='relu'))
  model.add(Dense(training_config["NUMBER_OF_CLASSES"], activation='softmax'))

  return model

def cheng_no_focal(training_config):
  model = Sequential(name="Cavanagh")
  model.add(Input(shape=(training_config["IMAGE_SIZE"], training_config["IMAGE_SIZE"], 3)))

  model.add(Conv2D(filters=32, kernel_size= 3, activation='relu', padding='same'))
  model.add(MaxPool2D(2))

  model.add(Conv2D(filters=64, kernel_size= 3, activation='relu', padding='same'))
  model.add(MaxPool2D(2))

  model.add(Conv2D(filters=128, kernel_size=2, activation='relu', padding='same'))
  model.add(MaxPool2D(2))
    
  model.add(Flatten())
  model.add(Dropout(0.5))

  model.add(Dense(1024, activation='relu'))
  model.add(Dense(1024, activation='relu'))

  model.add(Dropout(0.5))

  model.add(Dense(training_config["NUMBER_OF_CLASSES"], activation='softmax'))

  return model

def cheng_focal_10(training_config):
  model = Sequential(name="Cavanagh")
  model.add(Input(shape=(training_config["IMAGE_SIZE"], training_config["IMAGE_SIZE"], 3)))

  model.add(FocalConv(filters=32, kernel_size= 3, factor=10, activation='relu'))
  model.add(MaxPool2D(2))

  model.add(FocalConv(filters=64, kernel_size= 3, factor=10, activation='relu'))
  model.add(MaxPool2D(2))

  model.add(FocalConv(filters=128, kernel_size=2, factor=10, activation='relu'))
  model.add(MaxPool2D(2))
    
  model.add(Flatten())
  model.add(Dropout(0.5))

  model.add(Dense(1024, activation='relu'))
  model.add(Dense(1024, activation='relu'))

  model.add(Dropout(0.5))

  model.add(Dense(training_config["NUMBER_OF_CLASSES"], activation='softmax'))

  return model


models = [
    # Aspect 1
    {'name': 'cavanagh_no_focal', 'func': cavanagh_no_focal, 'starting_fold': 1},
    {'name': 'cavanagh_focal_10', 'func': cavanagh_focal_10, 'starting_fold': 1},
    {'name': 'cheng_no_focal', 'func': cheng_no_focal, 'starting_fold': 1},
    {'name': 'cheng_focal_10', 'func': cheng_focal_10, 'starting_fold': 1},


    # Aspect 2
    {'name': 'top_10_loss_4class_original', 'func': top_10_loss_4class_original, 'starting_fold': 1},
    {'name': 'top_10_loss_4class_all_focal', 'func': top_10_loss_4class_all_focal, 'starting_fold': 1},
    {'name': 'top_10_loss_4class_no_focal', 'func': top_10_loss_4class_no_focal, 'starting_fold': 1},
    {'name': 'best_loss_11class_original', 'func': best_loss_11class_original, 'starting_fold': 1},
    {'name': 'best_loss_11class_all_focal', 'func': best_loss_11class_all_focal, 'starting_fold': 1},
    {'name': 'best_loss_11class_no_focal', 'func': best_loss_11class_no_focal, 'starting_fold': 1},

]

perform_training(models, training_config)


# Other experiments

In [ ]:
@keras.saving.register_keras_serializable()
class FocalGaussConvExp(keras.layers.Layer):
    def __init__(self, filters, kernel_size, factor, activation):
        super().__init__()
        self.filters = filters
        self.kernel = kernel_size
        self.activation = activation
        self.factor = factor
        self.conv1 = Conv2D(filters=self.filters, kernel_size=self.kernel, activation=self.activation, padding="same")

    def build(self, input_shape):
        self.gauss = self.make_gaussian(size=input_shape[1], depth=self.filters, factor=self.factor)

    def call(self, x, training=False):
      x = self.conv1(x, training=training)
      batch_size = tf.shape(x)[0]
      xx = tf.repeat(self.gauss, repeats=batch_size, axis=0)
      x = multiply([x, xx])

      return x
    
    def get_config(self):
       return {
          "filters": self.filters,
          "kernel_size": self.kernel,
          "activation": self.activation,
          "factor": self.factor
       }

    def make_gaussian(self, size: int, depth: int, factor: float):
        fwhm = size // 2
        x = np.arange(0, size, 1, float)
        y = x[:,np.newaxis]
    
        x0 = y0 = size // 2
    
        sigma = 2 * fwhm ** 2
        gauss = np.exp(-1 * factor * (((x-x0) ** 2 / sigma)  + ((y-y0) ** 2 / sigma)))

        return tf.expand_dims(np.stack([gauss for _ in range(depth)], -1), axis=0)

@keras.saving.register_keras_serializable()
class FocalGaussConvRadius(keras.layers.Layer):
    def __init__(self, filters, kernel_size, factor, activation):
        super().__init__()
        self.filters = filters
        self.kernel = kernel_size
        self.activation = activation
        self.factor = factor
        self.conv1 = Conv2D(filters=self.filters, kernel_size=self.kernel, activation=self.activation, padding="same")

    def build(self, input_shape):
        self.gauss = self.make_gaussian(size=input_shape[1], depth=self.filters, factor=self.factor)

    def call(self, x, training=False):
      x = self.conv1(x, training=training)
      batch_size = tf.shape(x)[0]
      xx = tf.repeat(self.gauss, repeats=batch_size, axis=0)
      x = multiply([x, xx])

      return x
    
    def get_config(self):
       return {
          "filters": self.filters,
          "kernel_size": self.kernel,
          "activation": self.activation,
          "factor": self.factor
       }

    def make_gaussian(self, size: int, depth: int, factor: float):
        fwhm = size // 2 // factor
        x = np.arange(0, size, 1, float)
        y = x[:,np.newaxis]
    
        x0 = y0 = size // 2
    
        sigma = 2 * fwhm ** 2
        gauss = np.exp(-1 * (((x-x0) ** 2 / sigma)  + ((y-y0) ** 2 / sigma)))

        return tf.expand_dims(np.stack([gauss for _ in range(depth)], -1), axis=0)


@keras.saving.register_keras_serializable()
class SersicLogConv(keras.layers.Layer):
    def __init__(self, filters, kernel_size, factor, activation):
        super().__init__()
        self.filters = filters
        self.kernel = kernel_size
        self.activation = activation
        self.factor = factor
        self.conv1 = Conv2D(filters=self.filters, kernel_size=self.kernel, activation=self.activation, padding="same")

    def build(self, input_shape):
        self.gauss = self.make_gaussian(size=input_shape[1], depth=self.filters, factor=self.factor)

    def call(self, x, training=False):
      x = self.conv1(x, training=training)
      batch_size = tf.shape(x)[0]
      xx = tf.repeat(self.gauss, repeats=batch_size, axis=0)
      x = multiply([x, xx])

      return x
    
    def get_config(self):
       return {
          "filters": self.filters,
          "kernel_size": self.kernel,
          "activation": self.activation,
          "factor": self.factor
       }

    def make_gaussian(self, size: int, depth: int, factor: float):
        x,y = np.meshgrid(np.arange(size), np.arange(size))
    
        x0 = y0 = size // 2
        mod = Sersic2D(amplitude = 1, r_eff = size // 2, n=factor, x_0=x0, y_0=y0,
                    ellip=0, theta=1)
        
        img = mod(x,y)
        img = np.log(img)
        res = (img - img.min()) / (img.max() - img.min())

        return tf.expand_dims(np.stack([res for _ in range(depth)], -1), axis=0)

@keras.saving.register_keras_serializable()
class SersicPercentileConv(keras.layers.Layer):
    def __init__(self, filters, kernel_size, factor, activation):
        super().__init__()
        self.filters = filters
        self.kernel = kernel_size
        self.activation = activation
        self.factor = factor
        self.conv1 = Conv2D(filters=self.filters, kernel_size=self.kernel, activation=self.activation, padding="same")

    def build(self, input_shape):
        self.gauss = self.make_gaussian(size=input_shape[1], depth=self.filters, factor=self.factor)

    def call(self, x, training=False):
      x = self.conv1(x, training=training)
      batch_size = tf.shape(x)[0]
      xx = tf.repeat(self.gauss, repeats=batch_size, axis=0)
      x = multiply([x, xx])

      return x
    
    def get_config(self):
       return {
          "filters": self.filters,
          "kernel_size": self.kernel,
          "activation": self.activation,
          "factor": self.factor
       }

    def make_gaussian(self, size: int, depth: int, factor: float):
        x,y = np.meshgrid(np.arange(size), np.arange(size))
    
        x0 = y0 = size // 2
        mod = Sersic2D(amplitude = 1, r_eff = size // 2, n=factor, x_0=x0, y_0=y0,
                    ellip=0, theta=1)
        
        img = mod(x,y)
        img = np.log(img)
        res = PercentileInterval(90)(img)

        return tf.expand_dims(np.stack([res for _ in range(depth)], -1), axis=0)


@keras.saving.register_keras_serializable()
class FocalConvScalable(keras.layers.Layer):
    def __init__(self, filters, kernel_size, factor, activation):
        super().__init__()
        self.filters = filters
        self.kernel = kernel_size
        self.activation = activation
        self.factor = factor
        self.conv1 = Conv2D(filters=self.filters, kernel_size=self.kernel, activation=self.activation)

    def build(self, input_shape):
        conv_output_shape = input_shape[1] - self.kernel + 1 # no padding or stride
        self.gauss = self.make_gaussian(size=conv_output_shape, depth=self.filters, factor=self.factor)

    def call(self, x, training=False):
      x = self.conv1(x, training=training)
      batch_size = tf.shape(x)[0]
      xx = tf.repeat(self.gauss, repeats=batch_size, axis=0)
      x = multiply([x, xx])

      return x
    
    def get_config(self):
       return {
          "filters": self.filters,
          "kernel_size": self.kernel,
          "activation": self.activation,
          "factor": self.factor
       }

    def make_gaussian(self, size: int, depth: int, factor: float):
        """ Make a square gaussian kernel.

        size is the length of a side of the square
        fwhm is full-width-half-maximum, which
        can be thought of as an effective radius.
        """
        fwhm = size // 2
        x = np.arange(0, size, 1, float)
        y = x[:,np.newaxis]

        x0 = y0 = size // 2

        gauss = np.exp(-1 * factor * np.log(2) * ((x-x0) ** 2 + (y-y0) ** 2) / fwhm **2)

        return tf.expand_dims(np.stack([gauss for _ in range(depth)], -1), axis=0)


def create_model(training_config, focal_type, focal_factor, depth, filters, kernel, dropout, dense_depth, dense):
  model = Sequential(name="Cavanagh")
  model.add(Input(shape=(training_config["IMAGE_SIZE"], training_config["IMAGE_SIZE"], 3)))
    
  for _ in range(depth):
      print(f"Focal type: {focal_type}")
      match focal_type:
          case None:
              print("Added regular conv layer")
              model.add(Conv2D(filters=filters, kernel_size=kernel, activation='relu', padding="same"))
          case "focal":
              print(f"Added focal layer, focal factor: {focal_factor}")
              model.add(FocalConv(filters=filters, kernel_size= kernel, factor = focal_factor, activation='relu'))
          case "gaussian_exp":
              print(f"Added gaussian exp layer, focal factor: {focal_factor}")
              model.add(FocalGaussConvExp(filters=filters, kernel_size= kernel, factor = focal_factor, activation='relu'))
          case "gaussian_radius":
              print(f"Added gaussian radius layer, focal factor: {focal_factor}")
              model.add(FocalGaussConvRadius(filters=filters, kernel_size= kernel, factor = focal_factor, activation='relu'))
          case "sersic_log":
              print(f"Added sersic log layer, focal factor: {focal_factor}")
              model.add(SersicLogConv(filters=filters, kernel_size= kernel, factor = focal_factor, activation='relu'))
          case "sersic_perc":
              print(f"Added sersic percentile layer, focal factor: {focal_factor}")
              model.add(SersicPercentileConv(filters=filters, kernel_size= kernel, factor = focal_factor, activation='relu'))
          case _:
              raise Exception()
      model.add(BatchNormalization())
      model.add(MaxPool2D(2))


  model.add(Flatten())
  model.add(Dropout(dropout))

  for _ in range(dense_depth):
      model.add(Dense(dense, activation='relu'))

  model.add(Dense(training_config["NUMBER_OF_CLASSES"], activation='softmax'))

  return model

def create_common_model(training_config, focal_type, focal_factor):
    print(f"Experiment name: {experiment_name}")
    match experiment_name:
      case "Bin_SDSS":
          # Top 10 overall baselines
          # return create_model(training_config, focal_type = focal_type, focal_factor = focal_factor, depth = 4, filters = 32, kernel = 5, dropout = 0.7, dense_depth = 3, dense = 256)
          return create_model(training_config, focal_type = focal_type, focal_factor = focal_factor, depth = 4, filters = 64, kernel = 5, dropout = 0.7, dense_depth = 3, dense = 64)
      case "Bin_DES":
          # return create_model(training_config, focal_type = focal_type, focal_factor = focal_factor, depth = 4, filters = 64, kernel = 6, dropout = 0.7, dense_depth = 3, dense = 256)
          return create_model(training_config, focal_type = focal_type, focal_factor = focal_factor, depth = 5, filters = 256, kernel = 5, dropout = 0.7, dense_depth = 2, dense = 128)
      case "Bin_H_1":
          # return create_model(training_config, focal_type = focal_type, focal_factor = focal_factor, depth = 5, filters = 256, kernel = 6, dropout = 0.5, dense_depth = 3, dense = 256)
          return create_model(training_config, focal_type = focal_type, focal_factor = focal_factor, depth = 5, filters = 256, kernel = 7, dropout = 0.7, dense_depth = 3, dense = 256)
      case "Bin_H_2":
          # return create_model(training_config, focal_type = focal_type, focal_factor = focal_factor, depth = 3, filters = 256, kernel = 5, dropout = 0.7, dense_depth = 3, dense = 64)
          return create_model(training_config, focal_type = focal_type, focal_factor = focal_factor, depth = 5, filters = 256, kernel = 5, dropout = 0.3, dense_depth = 3, dense = 16)
      case "Bin_H_3":
          # return create_model(training_config, focal_type = focal_type, focal_factor = focal_factor, depth = 3, filters = 128, kernel = 7, dropout = 0.5, dense_depth = 3, dense = 32)
          return create_model(training_config, focal_type = focal_type, focal_factor = focal_factor, depth = 6, filters = 32, kernel = 5, dropout = 0.5, dense_depth = 3, dense = 16)

# logic: take the most common value for parameter in top 10. If there are multiple parameters with the same count, take the one with which better results were achieved
def baseline(training_config):
  return create_common_model(training_config, focal_type = None, focal_factor = 0)

def focal1(training_config):
  return create_common_model(training_config, focal_type = "focal", focal_factor = 1)

def focal2(training_config):
  return create_common_model(training_config, focal_type = "focal", focal_factor = 2)

def gauss_exp1(training_config):
  return create_common_model(training_config, focal_type = "gaussian_exp", focal_factor = 1)

def gauss_exp5(training_config):
  return create_common_model(training_config, focal_type = "gaussian_exp", focal_factor = 5)

def gauss_rad1(training_config):
  return create_common_model(training_config, focal_type = "gaussian_radius", focal_factor = 1)

def gauss_rad5(training_config):
  return create_common_model(training_config, focal_type = "gaussian_radius", focal_factor = 5)

def sersic_log1(training_config):
  return create_common_model(training_config, focal_type = "sersic_log", focal_factor = 1)

def sersic_log5(training_config):
  return create_common_model(training_config, focal_type = "sersic_log", focal_factor = 5)

def sersic_perc1(training_config):
  return create_common_model(training_config, focal_type = "sersic_perc", focal_factor = 1)

def sersic_perc5(training_config):
  return create_common_model(training_config, focal_type = "sersic_perc", focal_factor = 5)


def cavanagh_focal_1(training_config):
  model = Sequential(name="Cavanagh")
  model.add(Input(shape=(training_config["IMAGE_SIZE"], training_config["IMAGE_SIZE"], 3)))

  model.add(FocalConv(filters=32, kernel_size= 7, factor = 1, activation='relu'))
  model.add(BatchNormalization())
  model.add(MaxPool2D(2))

  model.add(FocalConv(filters=64, kernel_size= 5, factor = 1, activation='relu'))
  model.add(FocalConv(filters=64, kernel_size= 5, factor = 1, activation='relu'))
  model.add(BatchNormalization())
  model.add(MaxPool2D(2))

  model.add(FocalConv(filters=128, kernel_size=3, factor = 1, activation='relu'))
  model.add(BatchNormalization())
  model.add(MaxPool2D(2))

  model.add(Flatten())
  model.add(Dropout(0.5))

  model.add(Dense(256, activation='relu'))
  model.add(Dense(256, activation='relu'))
  model.add(Dense(training_config["NUMBER_OF_CLASSES"], activation='softmax'))

  return model

def cavanagh_focal_5(training_config):
  model = Sequential(name="Cavanagh")
  model.add(Input(shape=(training_config["IMAGE_SIZE"], training_config["IMAGE_SIZE"], 3)))

  model.add(FocalConv(filters=32, kernel_size= 7, factor = 5, activation='relu'))
  model.add(BatchNormalization())
  model.add(MaxPool2D(2))

  model.add(FocalConv(filters=64, kernel_size= 5, factor = 5, activation='relu'))
  model.add(FocalConv(filters=64, kernel_size= 5, factor = 5, activation='relu'))
  model.add(BatchNormalization())
  model.add(MaxPool2D(2))

  model.add(FocalConv(filters=128, kernel_size=3, factor = 5, activation='relu'))
  model.add(BatchNormalization())
  model.add(MaxPool2D(2))

  model.add(Flatten())
  model.add(Dropout(0.5))

  model.add(Dense(256, activation='relu'))
  model.add(Dense(256, activation='relu'))
  model.add(Dense(training_config["NUMBER_OF_CLASSES"], activation='softmax'))

  return model


def cheng_focal_1(training_config):
  model = Sequential(name="Cavanagh")
  model.add(Input(shape=(training_config["IMAGE_SIZE"], training_config["IMAGE_SIZE"], 3)))

  model.add(FocalConv(filters=32, kernel_size= 3, factor=1, activation='relu'))
  model.add(MaxPool2D(2))

  model.add(FocalConv(filters=64, kernel_size= 3, factor=1, activation='relu'))
  model.add(MaxPool2D(2))

  model.add(FocalConv(filters=128, kernel_size=2, factor=1, activation='relu'))
  model.add(MaxPool2D(2))
    
  model.add(Flatten())
  model.add(Dropout(0.5))

  model.add(Dense(1024, activation='relu'))
  model.add(Dense(1024, activation='relu'))

  model.add(Dropout(0.5))

  model.add(Dense(training_config["NUMBER_OF_CLASSES"], activation='softmax'))

  return model

def cheng_focal_5(training_config):
  model = Sequential(name="Cavanagh")
  model.add(Input(shape=(training_config["IMAGE_SIZE"], training_config["IMAGE_SIZE"], 3)))

  model.add(FocalConv(filters=32, kernel_size= 3, factor=5, activation='relu'))
  model.add(MaxPool2D(2))

  model.add(FocalConv(filters=64, kernel_size= 3, factor=5, activation='relu'))
  model.add(MaxPool2D(2))

  model.add(FocalConv(filters=128, kernel_size=2, factor=5, activation='relu'))
  model.add(MaxPool2D(2))
    
  model.add(Flatten())
  model.add(Dropout(0.5))

  model.add(Dense(1024, activation='relu'))
  model.add(Dense(1024, activation='relu'))

  model.add(Dropout(0.5))

  model.add(Dense(training_config["NUMBER_OF_CLASSES"], activation='softmax'))

  return model


### Models tuned specifically for datasets

def sdss_new_best_f1(training_config):
  model = Sequential(name="Cavanagh")
  model.add(Input(shape=(training_config["IMAGE_SIZE"], training_config["IMAGE_SIZE"], 3)))

  model.add(FocalConv(filters=64, kernel_size= 5, factor = 8, activation='relu'))
  model.add(BatchNormalization())
  model.add(MaxPool2D(2))

  model.add(FocalConv(filters=64, kernel_size= 7, factor = 3, activation='relu'))
  model.add(BatchNormalization())
  model.add(MaxPool2D(2))

  model.add(FocalConv(filters=16, kernel_size= 7, factor = 6, activation='relu'))
  model.add(BatchNormalization())
  model.add(MaxPool2D(2))
    
  model.add(Conv2D(filters=48, kernel_size=3, activation='relu', padding='same'))
  model.add(BatchNormalization())
  model.add(MaxPool2D(2))
    
  model.add(Flatten())
  model.add(Dropout(0.5))

  model.add(Dense(96, activation='relu'))
  model.add(Dense(128, activation='relu'))
  model.add(Dense(training_config["NUMBER_OF_CLASSES"], activation='softmax'))

  return model

def des_new_best_f1(training_config):
  model = Sequential(name="Cavanagh")
  model.add(Input(shape=(training_config["IMAGE_SIZE"], training_config["IMAGE_SIZE"], 3)))

  model.add(FocalConv(filters=32, kernel_size= 5, factor = 2, activation='relu'))
  model.add(BatchNormalization())
  model.add(MaxPool2D(2))

  model.add(FocalConv(filters=48, kernel_size= 7, factor = 2, activation='relu'))
  model.add(BatchNormalization())
  model.add(MaxPool2D(2))

  model.add(Conv2D(filters=48, kernel_size=7, activation='relu', padding='same'))
  model.add(BatchNormalization())
  model.add(MaxPool2D(2))
    
  model.add(FocalConv(filters=32, kernel_size= 5, factor = 5, activation='relu'))
  model.add(BatchNormalization())
  model.add(MaxPool2D(2))
    
  model.add(Flatten())
  model.add(Dropout(0.5))

  model.add(Dense(96, activation='relu'))
  model.add(Dense(96, activation='relu'))
  model.add(Dense(224, activation='relu'))
  model.add(Dense(training_config["NUMBER_OF_CLASSES"], activation='softmax'))

  return model

def h_1_new_best_f1(training_config):
  model = Sequential(name="Cavanagh")
  model.add(Input(shape=(training_config["IMAGE_SIZE"], training_config["IMAGE_SIZE"], 3)))

  model.add(Conv2D(filters=64, kernel_size= 7, activation='relu', padding='same'))
  model.add(BatchNormalization())
  model.add(MaxPool2D(2))

  model.add(Conv2D(filters=16, kernel_size=3, activation='relu', padding='same'))
  model.add(BatchNormalization())
  model.add(MaxPool2D(2))

  model.add(FocalConv(filters=48, kernel_size= 7, factor = 6, activation='relu'))
  model.add(BatchNormalization())
  model.add(MaxPool2D(2))
    
  model.add(Conv2D(filters=64, kernel_size=3, activation='relu', padding='same'))
  model.add(BatchNormalization())
  model.add(MaxPool2D(2))
    
  model.add(Flatten())
  model.add(Dropout(0.5))

  model.add(Dense(96, activation='relu'))
  model.add(Dense(training_config["NUMBER_OF_CLASSES"], activation='softmax'))

  return model

def h_2_new_best_f1(training_config):
  model = Sequential(name="Cavanagh")
  model.add(Input(shape=(training_config["IMAGE_SIZE"], training_config["IMAGE_SIZE"], 3)))

  model.add(Conv2D(filters=32, kernel_size= 5, activation='relu', padding='same'))
  model.add(BatchNormalization())
  model.add(MaxPool2D(2))
    
  model.add(FocalConv(filters=64, kernel_size= 5, factor = 7, activation='relu'))
  model.add(BatchNormalization())
  model.add(MaxPool2D(2))

  model.add(Conv2D(filters=32, kernel_size=5, activation='relu', padding='same'))
  model.add(BatchNormalization())
  model.add(MaxPool2D(2))
    
  model.add(Flatten())
  model.add(Dropout(0.5))

  model.add(Dense(256, activation='relu'))
  model.add(Dense(96, activation='relu'))
  model.add(Dense(224, activation='relu'))
  model.add(Dense(training_config["NUMBER_OF_CLASSES"], activation='softmax'))

  return model

def h_3_new_best_f1(training_config):
  model = Sequential(name="Cavanagh")
  model.add(Input(shape=(training_config["IMAGE_SIZE"], training_config["IMAGE_SIZE"], 3)))

  model.add(FocalConv(filters=64, kernel_size= 5, factor = 9, activation='relu'))
  model.add(BatchNormalization())
  model.add(MaxPool2D(2))

  model.add(Conv2D(filters=64, kernel_size= 5, activation='relu', padding='same'))
  model.add(BatchNormalization())
  model.add(MaxPool2D(2))

  model.add(Conv2D(filters=16, kernel_size=3, activation='relu', padding='same'))
  model.add(BatchNormalization())
  model.add(MaxPool2D(2))

  model.add(Conv2D(filters=32, kernel_size=5, activation='relu', padding='same'))
  model.add(BatchNormalization())
  model.add(MaxPool2D(2))
    
  model.add(Flatten())
  model.add(Dropout(0.5))

  model.add(Dense(64, activation='relu'))
  model.add(Dense(224, activation='relu'))
  model.add(Dense(224, activation='relu'))
  model.add(Dense(training_config["NUMBER_OF_CLASSES"], activation='softmax'))

  return model

